# CIRA-CIC-DoHBrw-2020 validation of the proposed detector

Goal 2 of the project: validate the proposed **time-distributed dilated conv → BiLSTM → residual-fusion** model on the public CIRA-CIC-DoHBrw-2020 dataset and compare with published results and with RF / XGBoost / MLP trained on the identical split.

Sections: **0 setup (Drive)** → **1 download dataset to Drive** → **2 inspect** → **3 per-flow protocol (T=1)** → **4 persistent-session windows (T=8) + ablation** → **5 binary task** → **6 collect results**.

Runtime: GPU (T4) recommended for sections 3–5.

In [ ]:
# ===== 0. Setup: Drive + repo =====
import os, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount failed/skipped:', e); os.makedirs('/content/drive/MyDrive', exist_ok=True)
DS = '/content/drive/MyDrive/doq-dataset'
CIRA = f'{DS}/cira-dohbrw-2020'          # dataset lives on Drive (persistent)
RES = f'{DS}/results/cira'                # results copied here
os.makedirs(CIRA, exist_ok=True); os.makedirs(RES, exist_ok=True)

REPO_URL = 'https://github.com/Cyberguru1/doq-tunnel-dataset.git'
if not os.path.isdir('/content/doq-tunnel-dataset'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/doq-tunnel-dataset'], check=True)
else:
    subprocess.run(['git', '-C', '/content/doq-tunnel-dataset', 'pull'], check=True)
os.chdir('/content/doq-tunnel-dataset')
subprocess.run(['pip', 'install', '-q', 'xgboost', 'tabulate', 'kagglehub'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('repo at', os.getcwd()); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# ===== 1. Download CIRA-CIC-DoHBrw-2020 CSVs to Drive =====
# Official host: https://cicresearch.ca/CICDataset/DoHBrw-2020/Dataset/ (now redirects to a UNB sign-up page).
# Fallback: Kaggle mirror of the ORIGINAL CSVs (dhoogla/cicdohbrw2020, 18 files). The dataset licence allows mirroring.
import glob, shutil
have = glob.glob(f'{CIRA}/**/*.csv', recursive=True) + glob.glob(f'{CIRA}/**/*.parquet', recursive=True)
if have:
    print(f'already have {len(have)} files in {CIRA}')
else:
    print('-- probing official host')
    print(subprocess.run(['curl', '-sIL', '--max-time', '30',
                          'https://cicresearch.ca/CICDataset/DoHBrw-2020/Dataset/CSVs/'],
                         capture_output=True, text=True).stdout[:600])
    print('-- trying Kaggle mirror via kagglehub')
    try:
        import kagglehub
        p = kagglehub.dataset_download('dhoogla/cicdohbrw2020')
        print('downloaded to', p)
        for f in glob.glob(f'{p}/**/*', recursive=True):
            if os.path.isfile(f):
                dst = os.path.join(CIRA, os.path.relpath(f, p)); os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(f, dst)
        print('copied to', CIRA)
    except Exception as e:
        print('kagglehub failed:', e)
print(subprocess.run(['du', '-sh', CIRA], capture_output=True, text=True).stdout)
for f in sorted(glob.glob(f'{CIRA}/**/*', recursive=True)):
    if os.path.isfile(f): print(f'{os.path.getsize(f)/1e6:8.1f} MB  {os.path.relpath(f, CIRA)}')

In [ ]:
# ===== 2. Inspect the files (columns, rows, labels) =====
import pandas as pd
for f in sorted(glob.glob(f'{CIRA}/**/*', recursive=True)):
    if not os.path.isfile(f): continue
    try:
        df = pd.read_parquet(f) if f.endswith('.parquet') else pd.read_csv(f, low_memory=False)
    except Exception as e:
        print('skip', f, e); continue
    lab = df['Label'].value_counts().to_dict() if 'Label' in df.columns else '-'
    print(f'{os.path.relpath(f, CIRA):40s} rows={len(df):8d} cols={df.shape[1]:3d} labels={lab}')
print(list(df.columns))

In [ ]:
# ===== 3. Tool identification, per-flow protocol (T=1, 80/10/10 stratified) + baselines on the same split =====
!python src/model/train.py --dataset cira --root {CIRA} --task tool --seq-len 1 --epochs 30 --baselines --out results/cira_tool_T1 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 4. Persistent-session windows (T=8 consecutive flows of one client->resolver session) + ablation (base variant) =====
!python src/model/train.py --dataset cira --root {CIRA} --task tool --seq-len 8 --epochs 30 --baselines --out results/cira_tool_T8 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA} --task tool --seq-len 8 --epochs 30 --variant base --out results/cira_tool_T8_base 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 5. Binary benign-DoH vs malicious-DoH (layer 2), T=1 and T=8 =====
!python src/model/train.py --dataset cira --root {CIRA} --task binary --seq-len 1 --epochs 20 --baselines --out results/cira_bin_T1 2>&1 | grep -v '^epoch'
!python src/model/train.py --dataset cira --root {CIRA} --task binary --seq-len 8 --epochs 20 --out results/cira_bin_T8 2>&1 | grep -v '^epoch'

In [ ]:
# ===== 6. Collect reports -> Drive =====
import shutil
for d in sorted(glob.glob('results/cira_*')):
    shutil.copytree(d, os.path.join(RES, os.path.basename(d)), dirs_exist_ok=True)
    print('\n' + open(os.path.join(d, 'report.md')).read())
print('saved to', RES)